# Overview of Training Loss per Component

![encoder_overview](../resources/v0_6/training_acpredictor_overview.png)

The BioJEPA-AC model has 3 major components for prediction, and 1 major evaluation head (there's also classification but we won't touch on that). Each of these components is trained separately and provides a unique component of the model's ability to predict cell states and changes based on a given perturbation. Since we have such a unique architecture, we'll walk through each component's loss evaluation and what is driving the model learning. 

In [1]:
import numpy as np
import torch.nn.functional as F
import torch

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

## Encoder 
The first major component in our model is the [cell state encoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_cell_state_encoder.ipynb). The cell state encoder learns the unified latent space into which our cells get embedded and across which perturbations move those representations. To train this space, we use an exponential-moving-average target encoder (teacher) in our self-supervised training loop. The target encoder helps the model avoid representation collapse without needing negative pairs, showing that a slow-moving target network alone provides enough signal asymmetry. 

During training, our context encoder (student) receives a masked version of the cell state while the target encoder sees the complete unmasked cell state and provides stable target latents for the context encoder to predict. We don't use gradients to update the teacher's weights, instead, as you'll see, we use the exponential moving average of the student's weights. This gives the context encoder a slowly-evolving, low-noise target that prevents representation collapse and smooths the training signal.

For our loss analysis we use a combination of L1 loss and VICReg loss. L1 loss drives the context encoder to accurately reconstruct the teacher's latents at masked positions, while VICReg loss prevents the representation space from collapsing by ensuring each feature dimension maintains variance and stays decorrelated from the others.

We'll start by staging the outputs of the context encoder and target encoder. We'll also have a mask index that shows which gene positions were masked per sample.

### Data Prep 

As part of the forward training pass for our context encoder, to ensure our inputs are masked, we calculate a set of mask indices. These will be used to ensure we only compare those gene positions. The forward pass then creates three latents:
1. `context_latents` created by the input of the masked sample data passing through the context encoder
2. `predicted_latents` created by the masked predictor processing the context latents
3. `target_latents` created by the input of the unmasked sample data passing through the target encoder.

Since we have a notebook explaining how this data is created, we'll focus on staging the latents and mask, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

In [2]:
batch = 2
num_genes = 8
embed_dim = 6   

**Mask Index**

Our first component is the boolean mask identifying which genes were masked for each sample in the batch. You'll see that the mask pattern is different per sample. During input, masking clears out the expression value for each masked gene, but for our loss calculation, it identifies the full embedding vector for each masked gene position.

In [3]:
mask_idx=torch.tensor([
    [True,True,True,False,True,False,True,False],
    [True,False,True,True,False,True,False,True]
])
mask_idx

tensor([[ True,  True,  True, False,  True, False,  True, False],
        [ True, False,  True,  True, False,  True, False,  True]])

**Context Latents**

These are the generated latents based on the masked input.  These are output by the context encoder. 

In [4]:
context_latents=torch.tensor([
    [[4.0,3.0,5.1,2.0,6.0,1.0],
    [5.0,4.0,5.1,3.0,7.0,2.0],
    [3.0,2.0,5.1,1.0,5.0,3.0],
    [6.0,5.0,5.1,4.0,8.0,1.0],
    [4.0,3.0,5.4,2.0,6.0,2.0],
    [5.0,4.0,5.1,3.0,7.0,1.0],
    [3.0,2.0,5.2,1.0,4.0,3.0],
    [7.0,6.0,5.1,5.0,9.0,2.0]],
    
    [[2.0,1.0,4.1,3.0,5.0,1.0],
    [6.0,5.0,5.1,4.0,8.0,2.0],
    [4.0,3.0,5.1,2.0,6.0,3.0],
    [3.0,2.0,5.1,1.0,4.0,1.0],
    [5.0,4.0,5.1,3.0,7.0,2.0],
    [7.0,6.0,5.1,5.0,9.0,3.0],
    [4.0,3.0,5.1,2.0,6.0,1.0],
    [2.0,1.0,5.3,1.0,3.0,2.0]],
])
context_latents

tensor([[[4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.4000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.2000, 1.0000, 4.0000, 3.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 2.0000]],

        [[2.0000, 1.0000, 4.1000, 3.0000, 5.0000, 1.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [2.0000, 1.0000, 5.3000, 1.0000, 3.0000, 2.0000]]])

**Predicted Latents**

These are the predicted latents based on the context latents.  These are output by the masked predictor.

In [5]:
predicted_latents=torch.tensor([
    [[4.2,2.8,5.3,1.8,5.7,1.1],
    [4.7,4.3,4.6,3.2,7.3,1.9],
    [3.3,1.7,5.4,0.8,4.8,3.2],
    [5.8,5.2,4.8,4.1,8.2,0.9],
    [3.8,3.2,5.2,2.1,6.3,2.1],
    [5.3,3.7,5.3,2.8,6.8,1.2],
    [2.7,2.3,4.7,1.2,4.2,2.8],
    [6.8,6.2,4.9,5.2,9.3,1.8]],

    [[2.3,0.7,5.3,2.8,4.7,1.2],
    [5.8,5.2,4.8,4.2,8.3,1.8],
    [3.7,3.3,4.7,2.2,6.3,2.8],
    [3.3,1.7,5.3,0.8,3.7,1.2],
    [4.8,4.2,4.8,3.2,7.3,1.8],
    [6.7,6.3,4.7,5.2,9.3,2.8],
    [4.3,2.7,5.3,1.8,5.7,1.2],
    [2.3,0.7,5.3,0.8,2.7,2.2]],
])
predicted_latents

tensor([[[4.2000, 2.8000, 5.3000, 1.8000, 5.7000, 1.1000],
         [4.7000, 4.3000, 4.6000, 3.2000, 7.3000, 1.9000],
         [3.3000, 1.7000, 5.4000, 0.8000, 4.8000, 3.2000],
         [5.8000, 5.2000, 4.8000, 4.1000, 8.2000, 0.9000],
         [3.8000, 3.2000, 5.2000, 2.1000, 6.3000, 2.1000],
         [5.3000, 3.7000, 5.3000, 2.8000, 6.8000, 1.2000],
         [2.7000, 2.3000, 4.7000, 1.2000, 4.2000, 2.8000],
         [6.8000, 6.2000, 4.9000, 5.2000, 9.3000, 1.8000]],

        [[2.3000, 0.7000, 5.3000, 2.8000, 4.7000, 1.2000],
         [5.8000, 5.2000, 4.8000, 4.2000, 8.3000, 1.8000],
         [3.7000, 3.3000, 4.7000, 2.2000, 6.3000, 2.8000],
         [3.3000, 1.7000, 5.3000, 0.8000, 3.7000, 1.2000],
         [4.8000, 4.2000, 4.8000, 3.2000, 7.3000, 1.8000],
         [6.7000, 6.3000, 4.7000, 5.2000, 9.3000, 2.8000],
         [4.3000, 2.7000, 5.3000, 1.8000, 5.7000, 1.2000],
         [2.3000, 0.7000, 5.3000, 0.8000, 2.7000, 2.2000]]])

**Target Latents**

These are the target latents based on the full input.  These are output by the target encoder. 

In [6]:
target_latents=torch.tensor([
    [[4.0,3.0,5.0,2.0,6.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [3.0,2.0,5.0,1.0,5.0,3.0],
    [6.0,5.0,5.0,4.0,8.0,1.0],
    [4.0,3.0,5.0,2.0,6.0,2.0],
    [5.0,4.0,5.0,3.0,7.0,1.0],
    [3.0,2.0,5.0,1.0,4.0,3.0],
    [7.0,6.0,5.0,5.0,9.0,2.0]],
    
    [[2.5,1.0,5.0,3.0,5.0,1.2],
    [6.0,5.0,5.0,4.0,8.0,2.0],
    [4.0,3.0,5.0,2.0,6.0,3.0],
    [3.0,2.0,5.0,1.0,4.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [7.0,6.0,5.0,5.0,9.0,3.0],
    [4.0,3.0,5.0,2.0,6.0,1.0],
    [2.0,1.0,5.0,1.0,3.0,2.0]],
])
target_latents

tensor([[[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 2.0000]],

        [[2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [2.0000, 1.0000, 5.0000, 1.0000, 3.0000, 2.0000]]])

### L1 Loss
Our first component in our total encoder training loss is L1 loss. L1 loss is the mean absolute difference between predicted and target values, penalizing errors proportionally to their magnitude without squaring them. We calculate it as: 
$$
\mathcal{L}_{L1} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_{i}| 
$$

We only compute L1 on masked positions because those are the genes the context encoder never saw, so predicting them tests whether the model has learned meaningful relationships between genes rather than just copying its input. Because of this, we actually compare the masked positions from our predicted latents against our target. 

We'll start by indexing out the masked positions. You'll see this masking removes the batch dimension and just returns the array of embeddings for the masked positions.

In [7]:
pred_masked = predicted_latents[mask_idx]
pred_masked.shape, pred_masked

(torch.Size([10, 6]),
 tensor([[4.2000, 2.8000, 5.3000, 1.8000, 5.7000, 1.1000],
         [4.7000, 4.3000, 4.6000, 3.2000, 7.3000, 1.9000],
         [3.3000, 1.7000, 5.4000, 0.8000, 4.8000, 3.2000],
         [3.8000, 3.2000, 5.2000, 2.1000, 6.3000, 2.1000],
         [2.7000, 2.3000, 4.7000, 1.2000, 4.2000, 2.8000],
         [2.3000, 0.7000, 5.3000, 2.8000, 4.7000, 1.2000],
         [3.7000, 3.3000, 4.7000, 2.2000, 6.3000, 2.8000],
         [3.3000, 1.7000, 5.3000, 0.8000, 3.7000, 1.2000],
         [6.7000, 6.3000, 4.7000, 5.2000, 9.3000, 2.8000],
         [2.3000, 0.7000, 5.3000, 0.8000, 2.7000, 2.2000]]))

In [8]:
target_masked = target_latents[mask_idx]
target_masked.shape, target_masked

(torch.Size([10, 6]),
 tensor([[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [2.0000, 1.0000, 5.0000, 1.0000, 3.0000, 2.0000]]))

**Calculate L1 Loss**

Now we're ready to calculate the loss on the masked positions between the predicted and the target. If you look closely, because of how we staged our data, most of our loss is fairly consistent across all positions resulting in a relatively low loss.

In [9]:
rec_loss = F.l1_loss(pred_masked, target_masked)
rec_loss

tensor(0.2467)

### Variance-Invariance-Covariance Regularization (VICReg) Loss
VICReg is a regularization technique that prevents representational collapse, where the encoder learns to map all inputs to similar outputs to trivially minimize reconstruction loss. The variance component ensures each feature dimension maintains healthy variance by penalizing any dimension that flattens below std of 1, while the covariance component penalizes correlations between feature dimensions. We calculate it as 
$$
\mathcal{L}_{VICReg} = \lambda_{std} \cdot \underbrace{\frac{1}{d} \sum_{j=1}^{d} \text{ReLU}(1 - \sigma_{j})}_{\text{std loss}} + \lambda_{cov} \cdot \underbrace{\frac{1}{d} \sum_{i \neq j} C_{ij}^{2}}_{\text{cov loss}}
$$

Where $\sigma_{j}$ is the standard deviation of feature dimension $j$ across samples, $C_{ij}$ is the $(i,j)$ element of the covariance matrix, and $\lambda_{std}$, $\lambda_{cov}$ are the weighting coefficients. This forces the model to spread information across the full embedding space rather than encoding redundant signals. We use a 25:1 ratio ($\lambda_{std}$ to $\lambda_{cov}$) to reflect that variance collapse (all features outputting the same value) is catastrophic and irreversible for training, while feature correlation is a softer inefficiency the model can tolerate and gradually correct.

Since VICReg is about ensuring we use the full embedding space, we compare the context and target latents across all positions.

In [10]:
std_coeff = 25.0
cov_coeff = 1.0

**Flatten Batch and Genes**

We start by merging the batch and gene dimensions so each gene position across both samples becomes a row. This reshapes our $[B, \text{num\_genes}, \text{embed\_dim}]$ tensors to $[B \times \text{num\_genes}, \text{embed\_dim}]$, giving us 16 rows to compute statistics over.

In [11]:
cont_x = context_latents.reshape(-1, embed_dim).float()
targ_y = target_latents.reshape(-1, embed_dim).float()
B = cont_x.shape[0]
B, cont_x.shape, cont_x, targ_y

(16,
 torch.Size([16, 6]),
 tensor([[4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.4000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.2000, 1.0000, 4.0000, 3.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 2.0000],
         [2.0000, 1.0000, 4.1000, 3.0000, 5.0000, 1.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [2.0000, 1.0000, 5.3000, 1.0000, 3.0000, 2.0000]]),
 tensor([[4.0000, 3.0000, 5

#### Standard Deviations Loss

Our first component to calculate is the standard deviation loss. For this we'll calculate the standard deviation of each embedding dimension across all gene positions, then sum the penalties from both the context and target latents. Notice that we're looking at the embedding channel since the goal is to balance how well it is distributing data across the channels. If it gets too low, we have an issue. 

We'll start by calculating the standard deviation for each latent. Notice that in particular our third channel is low. In fact, our target actually has no variation and is only pulled off of 0 variance because of our epsilon.

In [12]:
std_cont = torch.sqrt(cont_x.var(dim=0) + 0.0001)
std_cont

tensor([1.5864, 1.5864, 0.2747, 1.3602, 1.7702, 0.8063])

In [13]:
std_targ = torch.sqrt(targ_y.var(dim=0) + 0.0001)
std_targ

tensor([1.5408, 1.5864, 0.0100, 1.3602, 1.7702, 0.7933])

**Stdev Loss**

Now we're ready to compare our standard deviations to calculate a single loss that penalizes low-variance dimensions. Since the goal is to penalize deviations below 1, we first push anything above 1 negative via $1 - \text{stdev}$ and take the ReLU calculated as $\text{ReLU}(x) = \max(0, x)$. ReLU makes this a one-sided penalty since features with std below 1 get pushed up, but features with std above 1 are left alone. This prevents us from penalizing high-variance features. We then sum the context and target losses together.

In [14]:
stdloss_cont = (1 - std_cont)
stdloss_targ = (1 - std_targ)
stdloss_cont, stdloss_targ

(tensor([-0.5864, -0.5864,  0.7253, -0.3602, -0.7702,  0.1937]),
 tensor([-0.5408, -0.5864,  0.9900, -0.3602, -0.7702,  0.2067]))

In [15]:
stdloss_cont = F.relu(stdloss_cont)
stdloss_targ = F.relu(stdloss_targ)
stdloss_cont, stdloss_targ

(tensor([0.0000, 0.0000, 0.7253, 0.0000, 0.0000, 0.1937]),
 tensor([0.0000, 0.0000, 0.9900, 0.0000, 0.0000, 0.2067]))

In [16]:
stdloss_cont = torch.mean(stdloss_cont)
stdloss_targ = torch.mean(stdloss_targ)
stdloss_cont, stdloss_targ

(tensor(0.1532), tensor(0.1995))

In [17]:
std_loss = stdloss_cont + stdloss_targ
std_loss

tensor(0.3526)

#### Covariance Loss

Our second component penalizes correlations between embedding dimensions. We'll compute the covariance matrix across genes for each latent, then sum up the squared off-diagonal entries. The diagonal represents each dimension's variance with itself, which we already handle with std loss, so we only care about the off-diagonals. If two dimensions are correlated, the model is encoding redundant information and wasting capacity.

You'll notice that features 0 and 1 move in lockstep (feature 1 is always feature 0 minus 1), so their off-diagonal entry will be large. Features 3 and 4 also track each other. The covariance loss will push the model to decorrelate these pairs and spread information more independently across the embedding space.

We'll start by first mean-centering our latents.

In [18]:
cont_x = cont_x - cont_x.mean(dim=0)
cont_x

tensor([[-0.3750, -0.3750,  0.0250, -0.6250, -0.2500, -0.8750],
        [ 0.6250,  0.6250,  0.0250,  0.3750,  0.7500,  0.1250],
        [-1.3750, -1.3750,  0.0250, -1.6250, -1.2500,  1.1250],
        [ 1.6250,  1.6250,  0.0250,  1.3750,  1.7500, -0.8750],
        [-0.3750, -0.3750,  0.3250, -0.6250, -0.2500,  0.1250],
        [ 0.6250,  0.6250,  0.0250,  0.3750,  0.7500, -0.8750],
        [-1.3750, -1.3750,  0.1250, -1.6250, -2.2500,  1.1250],
        [ 2.6250,  2.6250,  0.0250,  2.3750,  2.7500,  0.1250],
        [-2.3750, -2.3750, -0.9750,  0.3750, -1.2500, -0.8750],
        [ 1.6250,  1.6250,  0.0250,  1.3750,  1.7500,  0.1250],
        [-0.3750, -0.3750,  0.0250, -0.6250, -0.2500,  1.1250],
        [-1.3750, -1.3750,  0.0250, -1.6250, -2.2500, -0.8750],
        [ 0.6250,  0.6250,  0.0250,  0.3750,  0.7500,  0.1250],
        [ 2.6250,  2.6250,  0.0250,  2.3750,  2.7500,  1.1250],
        [-0.3750, -0.3750,  0.0250, -0.6250, -0.2500, -0.8750],
        [-2.3750, -2.3750,  0.2250, -1.6

In [19]:
targ_y = targ_y - targ_y.mean(dim=0)
targ_y

tensor([[-0.4062, -0.3750,  0.0000, -0.6250, -0.2500, -0.8875],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500,  0.1125],
        [-1.4062, -1.3750,  0.0000, -1.6250, -1.2500,  1.1125],
        [ 1.5938,  1.6250,  0.0000,  1.3750,  1.7500, -0.8875],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500,  0.1125],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500, -0.8875],
        [-1.4062, -1.3750,  0.0000, -1.6250, -2.2500,  1.1125],
        [ 2.5938,  2.6250,  0.0000,  2.3750,  2.7500,  0.1125],
        [-1.9062, -2.3750,  0.0000,  0.3750, -1.2500, -0.6875],
        [ 1.5938,  1.6250,  0.0000,  1.3750,  1.7500,  0.1125],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500,  1.1125],
        [-1.4062, -1.3750,  0.0000, -1.6250, -2.2500, -0.8875],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500,  0.1125],
        [ 2.5938,  2.6250,  0.0000,  2.3750,  2.7500,  1.1125],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500, -0.8875],
        [-2.4062, -2.3750,  0.0000, -1.6

**Latent Covariance**

We can now compute the covariance matrix by taking the dot product of each feature dimension with every other feature dimension across samples. We normalize by $B - 1$ to apply Bessel's correction for unbiased estimation.

In [20]:
cov_cont = (cont_x.T @ cont_x) / (B - 1)
cov_cont.shape, cov_cont

(torch.Size([6, 6]),
 tensor([[ 2.5167,  2.5167,  0.1100,  1.8833,  2.7000,  0.1167],
         [ 2.5167,  2.5167,  0.1100,  1.8833,  2.7000,  0.1167],
         [ 0.1100,  0.1100,  0.0753, -0.0700,  0.0200,  0.0700],
         [ 1.8833,  1.8833, -0.0700,  1.8500,  2.2333, -0.0500],
         [ 2.7000,  2.7000,  0.0200,  2.2333,  3.1333,  0.0333],
         [ 0.1167,  0.1167,  0.0700, -0.0500,  0.0333,  0.6500]]))

In [21]:
cov_targ = (targ_y.T @ targ_y) / (B - 1)
cov_targ.shape, cov_targ

(torch.Size([6, 6]),
 tensor([[ 2.3740,  2.4375,  0.0000,  1.8958,  2.6583,  0.0621],
         [ 2.4375,  2.5167,  0.0000,  1.8833,  2.7000,  0.0850],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 1.8958,  1.8833,  0.0000,  1.8500,  2.2333, -0.0450],
         [ 2.6583,  2.7000,  0.0000,  2.2333,  3.1333,  0.0167],
         [ 0.0621,  0.0850,  0.0000, -0.0450,  0.0167,  0.6292]]))

**Off-diagonals**

We're now ready to pull all the values from the off-diagonals into a long list. This will then allow us to calculate the covariance loss across embedding dimensions.

In [22]:
n, m = cov_cont.shape
offd_cont = cov_cont.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_cont

tensor([ 2.5167,  0.1100,  1.8833,  2.7000,  0.1167,  2.5167,  0.1100,  1.8833,
         2.7000,  0.1167,  0.1100,  0.1100, -0.0700,  0.0200,  0.0700,  1.8833,
         1.8833, -0.0700,  2.2333, -0.0500,  2.7000,  2.7000,  0.0200,  2.2333,
         0.0333,  0.1167,  0.1167,  0.0700, -0.0500,  0.0333])

In [23]:
n, m = cov_targ.shape
offd_targ = cov_targ.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_targ

tensor([ 2.4375,  0.0000,  1.8958,  2.6583,  0.0621,  2.4375,  0.0000,  1.8833,
         2.7000,  0.0850,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.8958,
         1.8833,  0.0000,  2.2333, -0.0450,  2.6583,  2.7000,  0.0000,  2.2333,
         0.0167,  0.0621,  0.0850,  0.0000, -0.0450,  0.0167])

**Covariance Loss**

Now that we have our off-diagonal covariance, we're ready to calculate the loss. We first square the off-diagonal covariance values so both positive and negative correlations are penalized equally if they're the same magnitude, and so we quadratically punish bigger correlations. To get the penalty over all pairs, we then sum and normalize by dividing by dimension.

In [24]:
covl_cont = offd_cont.pow_(2).sum().div(embed_dim)
covl_cont

tensor(11.0202)

In [25]:
covl_targ = offd_targ.pow_(2).sum().div(embed_dim)
covl_targ

tensor(10.8135)

In [26]:
cov_loss = covl_cont + covl_targ
cov_loss

tensor(21.8336)

#### VICReg Loss

Now we're ready to calculate the actual VICReg loss. We scale each loss component (standard deviation loss and covariance loss) and then sum them together so we can tune how aggressively each type of collapse is penalized.

In [27]:
scale_std_loss = std_coeff * std_loss 
scale_std_loss

tensor(8.8158)

In [28]:
scale_cov_loss = cov_coeff * cov_loss
scale_cov_loss

tensor(21.8336)

In [29]:
reg_loss = scale_std_loss + scale_cov_loss
reg_loss

tensor(30.6495)

### Total Encoder Loss

Now that we have our L1 loss and our VICReg loss, we're ready to combine them. We use a similarity coefficient (`sim_coeff`) to scale the L1 reconstruction loss before adding it to VICReg. This balances the reconstruction objective against the regularization, ensuring the model learns accurate latent predictions while also maintaining a healthy embedding space.

In [30]:
sim_coeff = 25.0

In [31]:
rec_loss = sim_coeff * rec_loss
rec_loss 

tensor(6.1667)

In [32]:
encoder_loss = rec_loss + reg_loss
encoder_loss

tensor(36.8161)

### Target Encoder EMA Update

After we update the weights of the context encoder, we also need to update the target encoder or else the model will just converge and learn nothing. The update is based on the exponential moving average where we blend a portion of the student's current weights into the teacher with each step, using $\theta_{t} \leftarrow m \cdot \theta_{t} + (1 - m) \cdot \theta_{s}$ where $m$ is the EMA momentum. We use a default momentum of $0.995$ so the target encoder is moving significantly slower than the student. If you think through this, with a learning rate scheduler, as we get towards the later stages and the learning rate is annealing (aka dropping), the student is moving away from the teacher more slowly so the student and teacher should be coming closer together. Some model architectures anneal the momentum so by the end the student and teacher weights are equal.

## Action Composer
To handle different perturbations, we have the [action composer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_action_composer.ipynb). The composer projects different modalities of perturbations, based on their sequence and/or target, into a unified space so that we can pass them into the AC predictor to shift the cell state.  

To train the action composer we use contrastive learning to ensure that the sequence path and target path representations for the same perturbation are close while representations of different perturbations are far apart. For each perturbation, we do a separate sequence and target encoding, then use attention pooling to aggregate across perturbation slots into a single vector per sample. We L2-normalize both vectors, then calculate InfoNCE loss which is the cross-entropy over the cosine similarity matrix scaled by a fixed temperature.

InfoNCE loss teaches the model that the sequence encoding and target encoding of the same perturbation should be similar, while being dissimilar to every other perturbation in the batch. This forces the composer to learn that a given DNA sequence (or chemical structure) and its protein target are two views of the same underlying perturbation, so at inference time either pathway alone can produce a meaningful action latent.

We'll start by staging our attention pool outputs for the sequence and target.

### Data Prep 

As part of the forward training pass for our action composer, we only train on perturbations where we have both the sequence and target. While inference can support only having one or the other, to create our shared space we need both to complete contrastive learning. As part of the training forward pass, we create:
1. `z_seq` created by passing the sequence through the sequence encoder and attention pool
2. `z_target` created by passing the target through the target encoder and attention pool

Since we have a notebook explaining how this data is created, we'll focus on staging the latents, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss. If you look at our staged data, we've purposefully seeded some different values for our second and fourth perturbations to show how the loss is impacted when pairs don't match well.

In [33]:
batch = 4
pert_embed_dims = 8
temperature = 0.012

In [34]:
z_seq=torch.tensor([
    [3.0,1.0,4.0,0.5,2.0,1.0,3.5,0.5],
    [0.5,3.0,1.0,4.0,0.5,2.5,1.0,3.0],
    [2.0,0.5,1.0,3.0,4.0,0.5,2.0,1.0],
    [1.0,2.0,3.0,1.0,0.5,4.0,0.5,2.0],
])
z_seq.shape, z_seq

(torch.Size([4, 8]),
 tensor([[3.0000, 1.0000, 4.0000, 0.5000, 2.0000, 1.0000, 3.5000, 0.5000],
         [0.5000, 3.0000, 1.0000, 4.0000, 0.5000, 2.5000, 1.0000, 3.0000],
         [2.0000, 0.5000, 1.0000, 3.0000, 4.0000, 0.5000, 2.0000, 1.0000],
         [1.0000, 2.0000, 3.0000, 1.0000, 0.5000, 4.0000, 0.5000, 2.0000]]))

In [35]:
z_target=torch.tensor([
    [2.8,1.2,3.7,0.6,2.2,0.8,3.3,0.7],
    [6.7,2.8,1.2,3.8,0.6,0.3,1.2,2.8],
    [1.8,0.7,1.2,2.8,3.7,0.6,2.2,0.8],
    [2.2,1.5,1.1,3.1,3.9,1.5,2.2,1.1],
])
z_target.shape, z_target

(torch.Size([4, 8]),
 tensor([[2.8000, 1.2000, 3.7000, 0.6000, 2.2000, 0.8000, 3.3000, 0.7000],
         [6.7000, 2.8000, 1.2000, 3.8000, 0.6000, 0.3000, 1.2000, 2.8000],
         [1.8000, 0.7000, 1.2000, 2.8000, 3.7000, 0.6000, 2.2000, 0.8000],
         [2.2000, 1.5000, 1.1000, 3.1000, 3.9000, 1.5000, 2.2000, 1.1000]]))

### L2-Normalization

Our first step once we have the attention pool outputs is to normalize them. L2 normalization projects each vector onto the unit sphere so that the length of each vector is 1. This ensures that dot products between them become cosine similarities, removing magnitude differences and comparing only directional alignment. We calculate the normalization as:
$$
\hat{z} = \frac{z}{\|z\|_2} = \frac{z}{\sqrt{\sum_{i=1}^{d} z_i^{2}}}
$$

One item you'll notice is that all of the staged values decrease as we ensure that we create a length of 1.

In [36]:
z_seq = F.normalize(z_seq, dim=1)
z_seq.shape, z_seq

(torch.Size([4, 8]),
 tensor([[0.4536, 0.1512, 0.6047, 0.0756, 0.3024, 0.1512, 0.5292, 0.0756],
         [0.0765, 0.4588, 0.1529, 0.6118, 0.0765, 0.3824, 0.1529, 0.4588],
         [0.3357, 0.0839, 0.1678, 0.5035, 0.6713, 0.0839, 0.3357, 0.1678],
         [0.1678, 0.3357, 0.5035, 0.1678, 0.0839, 0.6713, 0.0839, 0.3357]]))

In [37]:
z_target = F.normalize(z_target, dim=1)
z_target.shape, z_target

(torch.Size([4, 8]),
 tensor([[0.4417, 0.1893, 0.5836, 0.0946, 0.3470, 0.1262, 0.5205, 0.1104],
         [0.7570, 0.3163, 0.1356, 0.4293, 0.0678, 0.0339, 0.1356, 0.3163],
         [0.3155, 0.1227, 0.2104, 0.4909, 0.6486, 0.1052, 0.3857, 0.1402],
         [0.3418, 0.2331, 0.1709, 0.4817, 0.6060, 0.2331, 0.3418, 0.1709]]))

### Temperature-Scaled Cosine Similarity Logits
Now that we've normalized, we're ready to calculate the cosine similarity between the sequence and target. We calculate it as: 
$$
\text{logits}_{ij} = \frac{\hat{z}_{seq,i} \cdot \hat{z}_{target,j}}{\tau}
$$
Where $\tau$ is the temperature. We include the division by temperature to sharpen the distribution by scaling up the cosine similarities before they go into cross-entropy, making the model more confident about which pairs match.

The output of this is a $[\text{Batch}, \text{Batch}]$ matrix of the similarity between each sequence and target. Since these are cosine similarities, the values will range from -1 to 1, where higher means more similar. This gives us the benefit of treating them as logits predicting the class. If you look at our second and fourth perturbations, you'll see that the index that corresponds to their position isn't the highest value in the row.

In [38]:
cos_sim = torch.matmul(z_seq, z_target.T) 
cos_sim.shape, cos_sim

(torch.Size([4, 4]),
 tensor([[0.9968, 0.6269, 0.7527, 0.7423],
         [0.4729, 0.6705, 0.6261, 0.7201],
         [0.7466, 0.6665, 0.9959, 0.9753],
         [0.6420, 0.5196, 0.4869, 0.5959]]))

In [39]:
logits = cos_sim / temperature
logits.shape, logits

(torch.Size([4, 4]),
 tensor([[83.0705, 52.2401, 62.7248, 61.8603],
         [39.4047, 55.8715, 52.1710, 60.0051],
         [62.2149, 55.5441, 82.9955, 81.2776],
         [53.5004, 43.2976, 40.5783, 49.6576]]))

### Similarity Labels
Since we're treating the cosine similarities as logits, we need a label to indicate which position in the sequence matches which target in our data.  This is as simple as labeling the diagonal. 

In [40]:
labels = torch.arange(logits.shape[0])
labels

tensor([0, 1, 2, 3])

### Cross Entropy
Now that we have our similarities as logits and labels, we can calculate the cross entropy. Cross-entropy measures how confidently the model assigns the highest probability (largest logit) to the correct matching pair (label) in each row of the similarity matrix, penalizing it when probability leaks to wrong pairs. It's calculated as:
$$
\mathcal{L}_{align} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp(\text{logits}_{ii})}{\sum_{j=1}^{B} \exp(\text{logits}_{ij})}
$$

Because of the issues we seeded into the second and fourth perturbations, the loss is quite high.

In [41]:
action_condition_loss = F.cross_entropy(logits, labels)
action_condition_loss

tensor(2.0447)

## Action Conditioned (AC) Predictor
How a cell state representation changes in the latent space is controlled by the [Action-Conditioned (AC) predictor](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_ac_predictor.ipynb). While the encoder learns a shared unified space for cell representations, the AC predictor learns how to traverse that space given a starting cell state and a perturbation. To train the predictor, we use locked context and target encoders. During this training, we allow the action composer to fine-tune at 1/10th the predictor's learning rate based on prediction accuracy.

During training, the AC predictor receives the masked context latents (encoded control cells) and the action latent, and generates the updated context latent. We also generate a target latent using the target encoder and the case cell. We use a mask annealing schedule to reduce masking over the last few epochs.

For our loss analysis we use a combination of beta-weighted Gaussian NLL loss and VICReg loss. Gaussian NLL loss drives the predictor to accurately predict the target encoder's latents of the perturbed cell at masked positions, while also learning calibrated uncertainty estimates through its variance output. We use beta annealing to gradually introduce the down-weighting of high-uncertainty predictions so that early on in training the model focuses on learning the mean. VICReg loss serves the same role as in encoder training, preventing the predicted latent space from collapsing.

We'll start by staging the outputs of the AC predictor and target encoding. We'll also have a mask index that shows which gene positions were masked per sample.

### Data Prep 

As part of the forward training pass for our AC predictor, to ensure our inputs are masked, we calculate a unique set of mask indices. These will be used to ensure we only compare those gene positions. The forward pass then creates three latents:
1. `pred_mu` representing the average latent of the perturbed cell generated by the AC predictor 
2. `pred_logvar` representing the average uncertainty of the perturbed cell generated by the AC predictor 
3. `target_latents` created by the input of the unmasked perturbed cell data passing through the target encoder.

Since we have a notebook explaining how this data is created, we'll focus on staging the latents and mask, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

**Mask Index**

Our first component is the boolean mask identifying which genes were masked for each sample in the batch. You'll see that the mask pattern is different per sample. During input, masking clears out the expression value for each masked gene, but for our loss calculation, it identifies the full embedding vector for each masked gene position.

In [42]:
mask_idx = torch.tensor([
    [True, False, True, True, False, True, False, True],
    [False, True, True, False, True, False, True, True],
])
mask_idx

tensor([[ True, False,  True,  True, False,  True, False,  True],
        [False,  True,  True, False,  True, False,  True,  True]])

**Predicted Mu**

This is the predicted cell latent based on the AC predictor shifting the cell state based on the perturbation.

In [43]:
pred_mu=torch.tensor([
    [[4.1,3.1,5.1,2.1,5.9,1.1],
    [4.9,4.1,4.9,3.1,7.1,2.1],
    [1.1,2.1,5.2,1.1,4.9,2.9],
    [6.1,5.1,5.1,4.1,8.1,1.1],
    [4.1,2.9,5.1,2.1,6.1,2.1],
    [5.1,4.1,5.1,3.1,7.1,1.1],
    [2.9,2.1,5.1,0.9,4.1,3.1],
    [7.1,6.1,5.1,5.1,9.1,2.1]],

    [[4.0,2.5,3.5,1.5,3.5,2.7],
    [6.1,5.1,5.1,4.1,8.1,2.1],
    [5.5,1.5,3.5,3.5,4.5,1.5],
    [4.5,0.5,3.5,2.5,2.5,2.5],
    [5.1,4.1,5.1,3.1,7.1,2.1],
    [5.5,4.5,3.5,3.5,7.5,1.5],
    [4.1,3.1,5.1,2.1,6.1,1.1],
    [3.5,2.5,3.5,2.5,1.5,3.5]],
])
pred_mu.shape, pred_mu

(torch.Size([2, 8, 6]),
 tensor([[[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
          [4.9000, 4.1000, 4.9000, 3.1000, 7.1000, 2.1000],
          [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
          [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
          [4.1000, 2.9000, 5.1000, 2.1000, 6.1000, 2.1000],
          [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
          [2.9000, 2.1000, 5.1000, 0.9000, 4.1000, 3.1000],
          [7.1000, 6.1000, 5.1000, 5.1000, 9.1000, 2.1000]],
 
         [[4.0000, 2.5000, 3.5000, 1.5000, 3.5000, 2.7000],
          [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
          [5.5000, 1.5000, 3.5000, 3.5000, 4.5000, 1.5000],
          [4.5000, 0.5000, 3.5000, 2.5000, 2.5000, 2.5000],
          [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
          [5.5000, 4.5000, 3.5000, 3.5000, 7.5000, 1.5000],
          [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
          [3.5000, 2.5000, 3.5000, 2.5000, 1.5000, 3.5000]]]))

**Predicted Log Variance**

This is the predicted uncertainty in the cell latent based on the AC predictor shifting the cell state based on the perturbation.

In [44]:
pred_logvar=torch.tensor([
    [[-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-0.4,-0.5,-0.4,-0.5,-0.4,-0.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-0.5,-0.4,-0.5,-0.4,-0.5,-0.4],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-0.4,-0.5,-0.4,-0.5,-0.4,-0.5]],

    [[-1.8,-2.0,-1.5,-2.0,-1.8,-2.0],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-2.0,-1.8,-2.0,-1.5,-2.0,-1.8],
    [-1.5,-2.0,-1.8,-2.0,-1.5,-2.0],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-2.0,-1.5,-2.0,-1.8,-2.0,-1.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-1.8,-2.0,-1.5,-2.0,-1.8,-2.0]],
])
pred_logvar.shape, pred_logvar

(torch.Size([2, 8, 6]),
 tensor([[[-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-0.4000, -0.5000, -0.4000, -0.5000, -0.4000, -0.5000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-0.5000, -0.4000, -0.5000, -0.4000, -0.5000, -0.4000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-0.4000, -0.5000, -0.4000, -0.5000, -0.4000, -0.5000]],
 
         [[-1.8000, -2.0000, -1.5000, -2.0000, -1.8000, -2.0000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-2.0000, -1.8000, -2.0000, -1.5000, -2.0000, -1.8000],
          [-1.5000, -2.0000, -1.8000, -2.0000, -1.5000, -2.0000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-2.0000, -1.5000, -2.0000, -1.8000, -2.0000, -1.5000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0

**Target Latents**

These are the target latents based on the full perturbed cell input. These are output by the target encoder.

*Note that we reuse the same staged values from the encoder section for simplicity. In actual training, these would be different since the encoder uses the unmasked control cell while full training uses the unmasked perturbed (case) cell.*

In [45]:
target_latents=torch.tensor([
    [[4.0,3.0,5.0,2.0,6.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [3.0,2.0,5.0,1.0,5.0,3.0],
    [6.0,5.0,5.0,4.0,8.0,1.0],
    [4.0,3.0,5.0,2.0,6.0,2.0],
    [5.0,4.0,5.0,3.0,7.0,1.0],
    [3.0,2.0,5.0,1.0,4.0,3.0],
    [7.0,6.0,5.0,5.0,9.0,2.0]],
    
    [[2.5,1.0,5.0,3.0,5.0,1.2],
    [6.0,5.0,5.0,4.0,8.0,2.0],
    [4.0,3.0,5.0,2.0,6.0,3.0],
    [3.0,2.0,5.0,1.0,4.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [7.0,6.0,5.0,5.0,9.0,3.0],
    [4.0,3.0,5.0,2.0,6.0,1.0],
    [2.0,1.0,5.0,1.0,3.0,2.0]],
])
target_latents.shape, target_latents

(torch.Size([2, 8, 6]),
 tensor([[[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
          [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
          [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
          [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
          [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
          [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
          [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
          [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 2.0000]],
 
         [[2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
          [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
          [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 3.0000],
          [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
          [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
          [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
          [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
          [2.0000, 1.0000, 5.0000, 1.0000, 3.0000, 2.0000]]]))

### Beta-Weighted Gaussian NLL Loss

Our first component in our total AC predictor training loss is Gaussian NLL loss. The AC predictor outputs both a mean ($\mu$) and log-variance ($\log\sigma^{2}$) per position, so the loss penalizes not just inaccurate predictions on $\mu$ but also miscalibrated confidence. If the model predicts a small variance but is wrong, the loss is severe. Without the beta weighting, the model could cheat by inflating uncertainty on all positions to reduce the squared error term. The beta weighting prevents this by scaling each element's loss by its detached variance raised to $\beta$, down-weighting predictions where the model claims high uncertainty. We anneal $\beta$ from 0 over the first 30% of training steps so the model first learns accurate mean predictions before being held accountable for calibrated confidence. We calculate the beta-weighted Gaussian NLL as:
$$
\mathcal{L}_{\beta\text{-NLL}} = \frac{1}{n} \sum_{i=1}^{n} \sigma_{i}^{2\beta} \cdot \frac{1}{2} \left( \log\sigma_{i}^{2} + \frac{(y_i - \mu_{i})^{2}}{\sigma_{i}^{2}} \right)
$$
We only compute Gaussian NLL on masked positions because those are the genes the predictor must reconstruct from the context encoder's partial view plus the perturbation action, testing whether the model has learned how a perturbation shifts the cell state. Because of this, we compare the masked positions from our predicted mean and log-variance against the target encoder's output of the perturbed cell.

We'll start by indexing out the masked positions. You'll see this masking removes the batch dimension and just returns the array of embeddings for the masked positions.

In [46]:
pred_mu_masked = pred_mu[mask_idx]
pred_mu_masked.shape, pred_mu_masked

(torch.Size([10, 6]),
 tensor([[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
         [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
         [7.1000, 6.1000, 5.1000, 5.1000, 9.1000, 2.1000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
         [5.5000, 1.5000, 3.5000, 3.5000, 4.5000, 1.5000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
         [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
         [3.5000, 2.5000, 3.5000, 2.5000, 1.5000, 3.5000]]))

In [47]:
pred_logvar_masked = pred_logvar[mask_idx]
pred_logvar_masked.shape, pred_logvar_masked

(torch.Size([10, 6]),
 tensor([[-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-0.4000, -0.5000, -0.4000, -0.5000, -0.4000, -0.5000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-2.0000, -1.8000, -2.0000, -1.5000, -2.0000, -1.8000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-1.8000, -2.0000, -1.5000, -2.0000, -1.8000, -2.0000]]))

In [48]:
target_masked = target_latents[mask_idx]
target_masked.shape, target_masked

(torch.Size([10, 6]),
 tensor([[4., 3., 5., 2., 6., 1.],
         [3., 2., 5., 1., 5., 3.],
         [6., 5., 5., 4., 8., 1.],
         [5., 4., 5., 3., 7., 1.],
         [7., 6., 5., 5., 9., 2.],
         [6., 5., 5., 4., 8., 2.],
         [4., 3., 5., 2., 6., 3.],
         [5., 4., 5., 3., 7., 2.],
         [4., 3., 5., 2., 6., 1.],
         [2., 1., 5., 1., 3., 2.]]))

**Calculate predicted variance**

Our model outputs log-variance so we need to convert it into variance. We do this by taking the exponential.

In [49]:
variance = torch.exp(pred_logvar_masked)
variance

tensor([[0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.6703, 0.6065, 0.6703, 0.6065, 0.6703, 0.6065],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.1353, 0.1653, 0.1353, 0.2231, 0.1353, 0.1653],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.1653, 0.1353, 0.2231, 0.1353, 0.1653, 0.1353]])

**Calculate Negative Log-Likelihood (NLL)**

Now that we have the variance, we can calculate the Gaussian NLL using the predicted cell state $\mu$, the target latent, and the variance. Since we have per-gene/embedding dimension variance, we can calculate the loss based off of that. As we calculate you can see that our seventh and last position have significantly higher loss than the other positions across all dimensions. Also, our second row has an issue in the very first position showing the value of having per-embedding position confidence.

In [50]:
nll = F.gaussian_nll_loss(
    pred_mu_masked.float(), 
    target_masked.float(), 
    variance, 
    reduction='none')
nll

tensor([[-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [ 2.7259, -0.1433, -0.2170, -0.1433, -0.2418, -0.1433],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [-0.1925, -0.2418, -0.1925, -0.2418, -0.1925, -0.2418],
        [-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [ 7.3127,  5.9059,  7.3127,  4.2919,  7.3127,  5.9059],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [ 5.9059,  7.3127,  4.2919,  7.3127,  5.9059,  7.3127]])

**Reconstruction Loss (Beta-weighted NLL)**

Now that we have our NLL, we're ready to add the beta weighting.  We generally use a low beta scaler on the variance. The beta-weighted NLL is called reconstruction loss because the model is reconstructing the teacher's latent representation at masked positions.

In [51]:
beta_nll = 0.10

In [52]:
rec_loss = (nll * variance.detach().pow(beta_nll)).mean() 
rec_loss

tensor(0.9483)

### Variance-Invariance-Covariance Regularization (VICReg) Loss

Our second component of the AC predictor loss is VICReg loss. This is calculated the same way as the encoder VICReg loss, except with the predicted cell state $\mu$ being compared against the target latent. Since the calculation is the same, I'll quickly go through the calculation.

*Note that we share the same coefficients.*

In [53]:
mu_x = pred_mu.reshape(-1, embed_dim).float()
targ_y = target_latents.reshape(-1, embed_dim).float()
B = mu_x.shape[0]
B, mu_x.shape, mu_x, targ_y

(16,
 torch.Size([16, 6]),
 tensor([[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
         [4.9000, 4.1000, 4.9000, 3.1000, 7.1000, 2.1000],
         [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
         [4.1000, 2.9000, 5.1000, 2.1000, 6.1000, 2.1000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
         [2.9000, 2.1000, 5.1000, 0.9000, 4.1000, 3.1000],
         [7.1000, 6.1000, 5.1000, 5.1000, 9.1000, 2.1000],
         [4.0000, 2.5000, 3.5000, 1.5000, 3.5000, 2.7000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
         [5.5000, 1.5000, 3.5000, 3.5000, 4.5000, 1.5000],
         [4.5000, 0.5000, 3.5000, 2.5000, 2.5000, 2.5000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
         [5.5000, 4.5000, 3.5000, 3.5000, 7.5000, 1.5000],
         [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
         [3.5000, 2.5000, 3.5000, 2.5000, 1.5000, 3.5000]]),
 tensor([[4.0000, 3.0000, 5

#### Standard Deviations Loss

In [54]:
std_mu = torch.sqrt(mu_x.var(dim=0) + 0.0001)
std_mu

tensor([1.4243, 1.4864, 0.7638, 1.1475, 2.1552, 0.7650])

In [55]:
std_targ = torch.sqrt(targ_y.var(dim=0) + 0.0001)
std_targ

tensor([1.5408, 1.5864, 0.0100, 1.3602, 1.7702, 0.7933])

**Stdev Loss**

In [56]:
stdloss_mu = (1 - std_mu)
stdloss_targ = (1 - std_targ)
stdloss_mu, stdloss_targ

(tensor([-0.4243, -0.4864,  0.2362, -0.1475, -1.1552,  0.2350]),
 tensor([-0.5408, -0.5864,  0.9900, -0.3602, -0.7702,  0.2067]))

In [57]:
stdloss_mu = F.relu(stdloss_mu)
stdloss_targ = F.relu(stdloss_targ)
stdloss_mu, stdloss_targ

(tensor([0.0000, 0.0000, 0.2362, 0.0000, 0.0000, 0.2350]),
 tensor([0.0000, 0.0000, 0.9900, 0.0000, 0.0000, 0.2067]))

In [58]:
stdloss_mu = torch.mean(stdloss_mu)
stdloss_targ = torch.mean(stdloss_targ)
stdloss_mu, stdloss_targ

(tensor(0.0785), tensor(0.1995))

In [59]:
std_loss = stdloss_mu + stdloss_targ
std_loss

tensor(0.2780)

#### Covariance Loss

In [60]:
mu_x = mu_x - mu_x.mean(dim=0)
mu_x

tensor([[-0.5062, -0.2375,  0.5062, -0.6750,  0.0750, -0.9375],
        [ 0.2938,  0.7625,  0.3063,  0.3250,  1.2750,  0.0625],
        [-3.5062, -1.2375,  0.6062, -1.6750, -0.9250,  0.8625],
        [ 1.4938,  1.7625,  0.5062,  1.3250,  2.2750, -0.9375],
        [-0.5062, -0.4375,  0.5062, -0.6750,  0.2750,  0.0625],
        [ 0.4938,  0.7625,  0.5062,  0.3250,  1.2750, -0.9375],
        [-1.7062, -1.2375,  0.5062, -1.8750, -1.7250,  1.0625],
        [ 2.4938,  2.7625,  0.5062,  2.3250,  3.2750,  0.0625],
        [-0.6062, -0.8375, -1.0938, -1.2750, -2.3250,  0.6625],
        [ 1.4938,  1.7625,  0.5062,  1.3250,  2.2750,  0.0625],
        [ 0.8938, -1.8375, -1.0938,  0.7250, -1.3250, -0.5375],
        [-0.1062, -2.8375, -1.0938, -0.2750, -3.3250,  0.4625],
        [ 0.4938,  0.7625,  0.5062,  0.3250,  1.2750,  0.0625],
        [ 0.8938,  1.1625, -1.0938,  0.7250,  1.6750, -0.5375],
        [-0.5062, -0.2375,  0.5062, -0.6750,  0.2750, -0.9375],
        [-1.1062, -0.8375, -1.0938, -0.2

In [61]:
targ_y = targ_y - targ_y.mean(dim=0)
targ_y

tensor([[-0.4062, -0.3750,  0.0000, -0.6250, -0.2500, -0.8875],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500,  0.1125],
        [-1.4062, -1.3750,  0.0000, -1.6250, -1.2500,  1.1125],
        [ 1.5938,  1.6250,  0.0000,  1.3750,  1.7500, -0.8875],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500,  0.1125],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500, -0.8875],
        [-1.4062, -1.3750,  0.0000, -1.6250, -2.2500,  1.1125],
        [ 2.5938,  2.6250,  0.0000,  2.3750,  2.7500,  0.1125],
        [-1.9062, -2.3750,  0.0000,  0.3750, -1.2500, -0.6875],
        [ 1.5938,  1.6250,  0.0000,  1.3750,  1.7500,  0.1125],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500,  1.1125],
        [-1.4062, -1.3750,  0.0000, -1.6250, -2.2500, -0.8875],
        [ 0.5938,  0.6250,  0.0000,  0.3750,  0.7500,  0.1125],
        [ 2.5938,  2.6250,  0.0000,  2.3750,  2.7500,  1.1125],
        [-0.4062, -0.3750,  0.0000, -0.6250, -0.2500, -0.8875],
        [-2.4062, -2.3750,  0.0000, -1.6

**Latent Covariance**

In [62]:
cov_mu = (mu_x.T @ mu_x) / (B - 1)
cov_mu.shape, cov_mu

(torch.Size([6, 6]),
 tensor([[ 2.0286,  1.4117, -0.0240,  1.5115,  1.9552, -0.5676],
         [ 1.4117,  2.2092,  0.5349,  1.2290,  2.8297, -0.4455],
         [-0.0240,  0.5349,  0.5833,  0.0245,  1.0035, -0.1564],
         [ 1.5115,  1.2290,  0.0245,  1.3167,  1.6380, -0.3763],
         [ 1.9552,  2.8297,  1.0035,  1.6380,  4.6447, -1.0237],
         [-0.5676, -0.4455, -0.1564, -0.3763, -1.0237,  0.5852]]))

In [63]:
cov_targ = (targ_y.T @ targ_y) / (B - 1)
cov_targ.shape, cov_targ

(torch.Size([6, 6]),
 tensor([[ 2.3740,  2.4375,  0.0000,  1.8958,  2.6583,  0.0621],
         [ 2.4375,  2.5167,  0.0000,  1.8833,  2.7000,  0.0850],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 1.8958,  1.8833,  0.0000,  1.8500,  2.2333, -0.0450],
         [ 2.6583,  2.7000,  0.0000,  2.2333,  3.1333,  0.0167],
         [ 0.0621,  0.0850,  0.0000, -0.0450,  0.0167,  0.6292]]))

**Off-diagonals**

In [64]:
n, m = cov_mu.shape
offd_mu = cov_mu.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_mu

tensor([ 1.4117, -0.0240,  1.5115,  1.9552, -0.5676,  1.4117,  0.5349,  1.2290,
         2.8297, -0.4455, -0.0240,  0.5349,  0.0245,  1.0035, -0.1564,  1.5115,
         1.2290,  0.0245,  1.6380, -0.3763,  1.9552,  2.8297,  1.0035,  1.6380,
        -1.0237, -0.5676, -0.4455, -0.1564, -0.3763, -1.0237])

In [65]:
n, m = cov_targ.shape
offd_targ = cov_targ.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_targ

tensor([ 2.4375,  0.0000,  1.8958,  2.6583,  0.0621,  2.4375,  0.0000,  1.8833,
         2.7000,  0.0850,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.8958,
         1.8833,  0.0000,  2.2333, -0.0450,  2.6583,  2.7000,  0.0000,  2.2333,
         0.0167,  0.0621,  0.0850,  0.0000, -0.0450,  0.0167])

**Covariance Loss**

In [66]:
covl_mu = offd_mu.pow_(2).sum().div(embed_dim)
covl_mu

tensor(7.7766)

In [67]:
covl_targ = offd_targ.pow_(2).sum().div(embed_dim)
covl_targ

tensor(10.8135)

In [68]:
cov_loss = covl_mu + covl_targ
cov_loss

tensor(18.5901)

#### VICReg Loss

In [69]:
scale_std_loss = std_coeff * std_loss 
scale_std_loss

tensor(6.9496)

In [70]:
scale_cov_loss = cov_coeff * cov_loss
scale_cov_loss

tensor(18.5901)

In [71]:
reg_loss = scale_std_loss + scale_cov_loss
reg_loss

tensor(25.5397)

### Total Predictor Loss

Now that we have our reconstruction loss and our VICReg loss, we're ready to combine them. We use a similarity coefficient (`sim_coeff`) to scale the reconstruction loss before adding it to VICReg. This balances the reconstruction objective against the regularization, ensuring the model learns accurate latent predictions while also maintaining a healthy embedding space.

In [72]:
rec_loss = sim_coeff * rec_loss
rec_loss 

tensor(23.7066)

In [73]:
predictor_loss = rec_loss + reg_loss
predictor_loss

tensor(49.2463)

## Linear Expression Decoder
Many of our evals rely on having the sample-level expression predicted. We do this using the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). To train the decoder, we compare the predicted change in expression (predicted delta) against the real change in expression (real delta).

For our loss analysis we calculate the mean squared error (MSE). MSE loss is the average of the squared differences between predicted and target values, penalizing larger errors disproportionately more than smaller ones.

We'll start by staging the outputs of the decoder as the calculated deltas.

### Data Prep 

As part of the forward training pass for our linear expression decoder, we go through the full BioJEPA-AC forward pass, then through the decoder. We calculate expression predictions for the control cell and the perturbed cell, then use those to calculate: 
1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression from the predicted control expression. We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

Since we have a notebook explaining how this data is created, we'll focus on staging the deltas, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

In [74]:
batch = 2
num_genes = 8  

**Predicted Delta**

The predicted delta is the difference between the predicted expression of the perturbed cell and the control cell.

In [75]:
pred_delta=torch.tensor([
    [0.6,-0.2,1.1,-0.7,0.2,-1.4,0.8,-0.1],
    [0.2,0.3,-0.4,-0.3,1.1,-0.5,0.2,0.3],
])
pred_delta.shape, pred_delta

(torch.Size([2, 8]),
 tensor([[ 0.6000, -0.2000,  1.1000, -0.7000,  0.2000, -1.4000,  0.8000, -0.1000],
         [ 0.2000,  0.3000, -0.4000, -0.3000,  1.1000, -0.5000,  0.2000,  0.3000]]))

**Real Delta**

The real delta is the difference between the real expression of the perturbed cell and the control cell.

In [76]:
real_delta=torch.tensor([
    [0.5,-0.3,1.2,-0.8,0.1,-1.5,0.7,-0.2],
    [1.0,-3.5,0.8,-1.2,0.3,0.6,-0.9,1.5],
])

real_delta.shape, real_delta

(torch.Size([2, 8]),
 tensor([[ 0.5000, -0.3000,  1.2000, -0.8000,  0.1000, -1.5000,  0.7000, -0.2000],
         [ 1.0000, -3.5000,  0.8000, -1.2000,  0.3000,  0.6000, -0.9000,  1.5000]]))

### Mean Squared Error (MSE) Loss

We're now ready to calculate the mean squared error loss. The MSE loss is the average of the squared differences between predicted and target values. We calculate it as:
$$
\mathcal{L}_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_{i})^{2}
$$

The MSE penalizes larger errors disproportionately more than smaller ones. Because of this, you'll see that the second sample in our staged data has a significantly larger error because of the second gene position.

In [77]:
decoder_loss = F.mse_loss(pred_delta, real_delta)
decoder_loss

tensor(1.3694)